# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries and datasets

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
#import sys
#sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
#import utilities
#from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
from scipy import stats
from collections import defaultdict

In [ ]:
#Daily chlorophyll data and regional zarr files
daily_data = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_D8.zarr')
MABN = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_D8.zarr')
GB = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_D8.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_D8.zarr')
GOME = xr.open_zarr(r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_D8.zarr')

In [ ]:
#Region shapefiles
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

In [ ]:
def threshold_value(thld=0.1, path=None):
    """
    Calculates the threshold value for chlorophyll-a based on a median baseline provided by the regional climatology.

    If no file path is provided, the path defaults to grabbing and reading the annual climatology file for the Northeast Shelf (NES) region. 
    The threshold is calculated by finding the percentage above the climatological CHL median for each pixel in the region.

    Args:
        thld (float, optional): The fraction value of the percentage above the median. This value defaults to 0.1 (10%).
        path (str, optional): The path to netCDF file used to calculate the threshold value. Defaults to None.

    Returns:
        xarray.DataArray: A spatial array containing the threshold values for each coordinate based on the median CHL value
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return thld_value

In [ ]:
def spatial_threshold_value(shapefile_geometry=None,thld=0.1,path=None,regions=None,region_col='Region',ordered_region_names=None,default_shapefile_path='https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip'):
    """
    Creates a threshold value for a spatially averaged area.

    Using the threshold value, this function creates a spatially averaged threshold value for a region for use in other analysis.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, optional): Shapefile of the region in question. Defaults to None
        thld (float, optional): Percentage for threshold calcultion. Defaults to 0.1.
        path (str, optional): Path to climatology file. Defaults to None.
        regions (list or str, optional): A subset of region names to filter by. Defaults to None
        region_col (str, optional): The column name of the shapefile containing region names.
        ordered_region_names (list, optional): The list of names of the region in the shapefile if not already a column in the shapefile. Defaults to None
        default_shapefile_path (str, optional): Path to the default shapefile
    
    Returns:
        float. Value of the regionally averaged chlorophyll threshold.
    """
    # STEP 1: Load the shapefile geometry
    if shapefile_geometry is None:
        shapefile = gpd.read_file(default_shapefile_path)
        ordered_regions = ['Middle Atlantic Bight South','Middle Atlantic Bight North', 'Georges Bank', 'Gulf of Maine West', 'Gulf of Maine East']
        shapefile['Region'] = ordered_regions
    else:
        shapefile = shapefile_geometry.copy()
    if shapefile.crs is None:
        shapefile = shapefile.set_crs("EPSG:4326")
    else:
        shapefile = shapefile.to_crs("EPSG:4326")
    if ordered_region_names is not None:
        if len(ordered_region_names) != len(shapefile):
            raise ValueError("The list of names provided does not match the number of rows in the shapefile")
        shapefile['Region'] = ordered_region_names
        region_col = 'Region'

    # STEP 2: Subset the shapefile
    if regions is not None:
        if isinstance(regions,str):
            regions = [regions]
        shapefile = shapefile[shapefile[region_col].isin(regions)]
        if shapefile.empty:
            raise ValueError(f"None of the provided regions {regions} were found in the shapefile")
    
    # STEP 3: Load and prepare threshold data
    threshold = threshold_value(thld=thld,path=path)
    threshold.rio.write_crs("EPSG:4326",inplace=True)
    threshold.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    
    # STEP 4: Build the Dataset
    results = []
    for _, row in shapefile.iterrows():
        region_name = row[region_col]
        region_geometry = [mapping(row.geometry)]
        try:
            clipped_thld = threshold.rio.clip(region_geometry, shapefile.crs, drop=True)
            clipped_thld = clipped_thld.mean(dim=['lat','lon']).item()
            results.append({'Region':region_name, 'Threshold': clipped_thld})
        except Exception as e:
            print(f"Skipping {region_name} due to processing error")
            results.append({'Region': region_name, 'Threshold': None})

    return pd.DataFrame(results)

In [ ]:
#Calculating threshold and median values for each region

threshold_10 = spatial_threshold_value(path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
median_climatology = spatial_threshold_value(thld=0, path=r'C:\Users\grace\OneDrive\Documents\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')

MABS_thld = threshold_10['Threshold'][0]
MABN_thld = threshold_10['Threshold'][1]
GB_thld = threshold_10['Threshold'][2]
GOMW_thld = threshold_10['Threshold'][3]
GOME_thld = threshold_10['Threshold'][4]
MABS_median = median_climatology['Threshold'][0]
MABN_median = median_climatology['Threshold'][1]
GB_median = median_climatology['Threshold'][2]
GOMW_median = median_climatology['Threshold'][3]
GOME_median = median_climatology['Threshold'][4]

In [ ]:
def bounding_data(dataset,shapefile_geometry):
    """
    Regionally subsets a dataset for general analysis.

    This function takes a shapefile geometry and subsets a larger dataset to only include data within the shapefile. The data is averaged along the lat and lon dimensions.

    Args:
        dataset (xarray.Dataset, required): General dataset in question. No defaults
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults

    Returns:
        xarray.DataArray. The arrays of spatially sliced data.
    """
    dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    dataset.rio.write_crs("epsg:4326", inplace=True)
    clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile_geometry.crs, drop=True)
    regional_year = clipped_daily.CHL_median.mean(dim=['lat','lon'])
    return regional_year

In [ ]:
def smoothing_data(dataset=None, path=None, var_name='CHL_median', dim='time', method="SavGol", window=15, poly=3, deriv=0, frac=0.00117):
    """
    Smoothes the raw chlorphyll-a data using a specific smoothing technique.

    If no method is provided, the default is the Savistky-Golay technique which has default parameters of a 15 day window and a polyorder of 3.
    If method is provided as "lowess", the frac value defaults to 0.00117, equivalent of a 12 day window on a 27 year time series.
    If no dataset path is provided, the function searches for D8 CHL files for the NES region. Currently, it opens the zarr file, but can be uncomment to open netCDFs.

    Args:
        dataset (xarray.Dataset, optional): A spatially averaged dataset. Default is None
        path (str, optional): Path to a dataset. Defaults to daily D8 data for the full time series.
        var_name (str, optional): Name of variable of interest in the dataset. Defaults to 'CHL_median'
        dim (str, optional): The dimension to interpolate and smooth across. Defaults to 'time'
        method (str, optional): Smoothing technique applied. Defaults to "SavGol" but can also receive "lowess".
        window (int, optional): Window for SavGol smoothing. Default is 15
        poly (int, optional): polyorder for SavGol smoothing. Default is 3
        deriv (int, optional): Derivative of SavGol function. 0 provides smoothed data and 1 provides the first derivative. Defaults to 0 
        frac (float, optional): Frac value for lowess smoothing. Only necessary for using lowess smoothing. Default is 0.00117

    Returns:
        numpy.ndarray. Array of smoothed chlorophyll data values.
    """
    if dataset is not None:
        data = dataset
    elif path is None:
        data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_combined.zarr')
    else:
        data = xr.open_mfdataset(path)
    
    if isinstance(data, xr.Dataset):
        chl = data[var_name]
    else:
        chl = data

    #Savitsky-Golay smoothing
        #Requires no NaN values so it uses a linear interpolation to fill values.
    if method == "SavGol":
        data_filled = (chl.interpolate_na(dim=dim, method='linear', fill_value='extrapolate'))
        #Does the smoothing for each point in the spatial data along the time dimension. Should work as long as there is a time dimension.
        smoothed_CHL = xr.apply_ufunc(
            scipy.signal.savgol_filter,
            data_filled,
            kwargs={
                "window_length": window,
                "polyorder": poly,
                "deriv": deriv,
                "axis": -1,
            },
            input_core_dims=[[dim]],
            output_core_dims=[[dim]],
            dask='parallelized',
            output_dtypes=[chl.dtype]
        )
        smoothed_median = smoothed_CHL.where(~np.isnan(chl))
    #LOWESS smoothing
        #LOWESS function handles NaN values so no interpolation is needed.
    elif method == "lowess":
        def apply_lowess(y):
            mask = ~np.isnan(y)
            if mask.sum() < 3:
                return np.full_like(y, np.nan)
            smoothed = sm.nonparametric.smoothers_lowess.lowess(
                endog=y[mask],
                exog=np.arange(len(y))[mask],
                frac=frac,
                return_sorted=False
            )
            out = np.full_like(y, np.nan, dtype=np.float64)
            out[mask] = smoothed
            return out
        smoothed_median = xr.apply_ufunc(
            apply_lowess,
            chl,
            input_core_dims=[[dim]],
            output_core_dims=[[dim]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[chl.dtype]
        )
    else:
        raise ValueError("Error: Must specify smoothing technique")
    return smoothed_median

In [ ]:
def bloom_peak_detection_1D(clipped_thld,dataset,window_for_peak=10,days=14,prm=0.1):
    """
    Detects all peak chlorophyll values that exceed the climatological threshold for a 1-D array

    This function uses the smoothed chlorophyll data, identified peak values, and then masks that data to include only peaks that exceed the threshold set by the climatology.
    This function uses the find_peaks function from scipy, as well as the threshold_value() function and smoothing_data() function. 
    The clipped_thld variable is created in the rolling_peak_window function or as a global variable.

    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold value for the region of interest. No defaults
        dataset (xarray.Dataset, required): Already smoothed dataset. Defaults to None
        days (int, optional): Distance variable for scipy find_peaks. Distance allowed between consecutive peaks. Default is 14
        prm (float, optional): Prominence variable for find_peaks. Percent above the other peaks to be considered a peak. Default is 0.1
    
    Returns:
        List. List of days since the start of the dataset where the chlorophyll peaked and was above the threshold.
    """
    # STEP 1: Define the dataset. Uses a smoothed dataset (if provided). Else, it smooths the raw data provided for the region of interest.
    smoothed_CHL = np.asarray(dataset).ravel()
    if np.isnan(smoothed_CHL).all():
        return np.zeros(smoothed_CHL.shape, dtype=bool)
    thld = float(np.asarray(clipped_thld).flat[0])
    # STEP 2: Find chlorophyll peaks with find peaks function.
        # Default of 14 days for distance was chosen after testing distances from 7-31
        # Default prominence of 0.1 captures major blooms while ignoring small peaks from daily fluctuations/sensor noise. Tested values in range of 0.01 - 0.2. 
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    chl_peaks = []
    chl_series = pd.Series(smoothed_CHL) #Turns chlorophyll values into a pandas series

    # STEP 3: Create a Boolean list for values that surpass/do not exceed the set threshold.
    is_above_threshold = chl_series>thld #Creates a true and false list. True if the value exceeds the threshold.

    # STEP 4: Searches Boolean list for places where the value switches from True to False (or False to True)
    change_from_prev_day = is_above_threshold != is_above_threshold.shift() #Checks if there is a change from previous day

    # STEP 5: Create streak IDs for each event and group events with the same ID together
    streak_IDs = change_from_prev_day.cumsum() #Creates ID for each event (New ID starts when the Boolean value changes. If no change, the ID is the same for that day)
    streak_lengths = is_above_threshold.groupby(streak_IDs).transform('sum') #Groups events together with the same ID and calculates the number of days that share that ID
    
    # STEP 6: Check to ensure the peak is above the threshold and check to see if its streak ID is >= to the defined window_for_peak.
        # If both conditions are true, we add it to the chl_peaks list. If one or both is not met, the peak is discarded.
    for peak in chl_peak_loc:
        peak_above_threshold = is_above_threshold[peak] #Checks that the peak is above the threshold
        peak_length = streak_lengths[peak]>=window_for_peak #Checks that the chlorophyll values remain above the threshold for a specified window
        if peak_above_threshold and peak_length:
            chl_peaks.append(peak)
    mask = np.zeros(len(dataset), dtype=bool)
    mask[chl_peaks] = True
    return mask

In [ ]:
def bloom_peak_detection(data, clipped_thld, var_name='smoothed_CHL', time_dim='time', window_for_peak=10, days=14, prm=0.1, return_indices_for_1d=True):
    """
    Identifies the chlorophyll peaks in the dataset.

    Args:
        data (xr.Dataset, required): The dataset for the chlorophyll data. No defaults
        clipped_thld (float, required): The threshold value for the pixels/region. No defaults
        var_name (str, optional): The name of the smoothed chlorophyll variable in the dataset. Defaults to 'smoothed_CHL'
        time_dim (str, optional): The time dimension name in the dataset. Defaults to 'time'
        window_for_peak (int, optional): The number of days the chlorophyll must be exceeding the threshold to be considered a peak. Defaults to 10
        days (int, optional): The minimum number of days between peaks. Defaults to 14
        prm (float, optional): The minimum prominence of the peaks compared to surrounding chlorophyll. Defaults to 0.1
        return_indices_for_1d (Bool, optional): Determines whether the function also returns a list of integers or a xr.Dataset with dimensions and coordinates. Defaults to True

    Returns:
        List, xr.DataArray, pd.Series, or np.ndarray: Output type depends on 'return_indices_for_1d' and type of 'chl':
            - List: List of integer indices where chlorophyll peaked above the threshold. Returned if 'return_indices_for_1d' is True
            - xr.DataArray: Boolean mask matching the original coordinates and dimensions of `chl`. Returned if `return_indices_for_1d` is False and `chl` is an xarray DataArray.
            - pd.Series: Boolean mask matching the original index of `chl`. Returned if `return_indices_for_1d` is False and `chl` is a pandas Series.
            - np.ndarray: Raw 1-D boolean array mask. Returned if `return_indices_for_1d` is False and `chl` is any other array type.
    """
    #Opens datasets. Can take a xarray Dataset, DataArray, pandas DataFrame, Series, or numpy array. 
    if isinstance(data, (xr.Dataset, pd.DataFrame)):
        #If the data is a dataset or dataframe, the user must specify the variable name.
        if var_name in data:
            chl = data[var_name]
        else:
            raise ValueError("Must specify variable name")
    elif isinstance(data, (xr.DataArray, pd.Series, np.ndarray)):
        chl = data
    else:
        raise TypeError("Unsupported data type")
    # If the data is multi-dimensional (spatial), then it applies the bloom_peak_detection_1D function along the time dimension for each pixel.
    if getattr(chl, 'ndim', 1) > 1:
        peak_mask = xr.apply_ufunc(
            bloom_peak_detection_1D,
            chl,
            clipped_thld,
            input_core_dims=[[time_dim], []],
            output_core_dims=[[time_dim]],
            kwargs={
                'window_for_peak': window_for_peak,
                'days': days,
                'prm': prm
            },
            vectorize=True,
            dask='parallelized',
            output_dtypes=[bool],
            dask_gufunc_kwargs={'allow_rechunk':True} #Allows for rechunking of the data to optimize parallel processing
        )
        return peak_mask.transpose(*chl.dims)
    # If the data is one dimensional, it applies the bloom_peak_detection_1D function, returning either a list of indices or a boolean mask.
    else:
        if isinstance(chl, (xr.DataArray, pd.Series)):
            values = chl.values
        else:
            values = np.asarray(chl)
        thld_values = float(clipped_thld.values) if isinstance(clipped_thld, xr.DataArray) else clipped_thld
        mask_1d = bloom_peak_detection_1D(
            dataset=values,
            clipped_thld=thld_values,
            window_for_peak=window_for_peak,
            days=days,
            prm=prm
        )
        if return_indices_for_1d: #Returns a list of boolean values if False.
            return np.where(mask_1d)[0].tolist()
        else:
            if isinstance(chl, xr.DataArray):
                return xr.DataArray(mask_1d, coords=chl.coords, dims=chl.dims)
            elif isinstance(chl, pd.Series):
                return pd.Series(mask_1d, index=chl.index)
            return mask_1d

In [ ]:
def bloom_event_detection(clipped_thld,chl_peaks_list,dataset,var_name='smoothed_CHL',event_distance=21,peak_window=10):
    """
    This function finds peaks in the chlorophyll-a time series and then groups together peaks in the same event based on proximity.

    If there are no peaks, the function returns an empty list.
    Using a list of chlorophyll peaks from the bloom_peak_detection function (previously calculated), the function searches for peaks in close proximity.
    A ten day rolling window is created for each peak to see if the chlorophyll value drop below the climatological threshold. If it does, the loop breaks.
    If peaks are too close together or the chlorophyll value does not drop below the threshold, they are considered one event. If these conditions are not met, they are separate events.

    Args:
        clipped_thld (variable, required): Threshold value for the region of interest. No defaults
        chl_peaks_list (list, required): Pre-calculated list of chlorophyll peaks for dataset. No defaults
        dataset (variable, required): Already smoothed dataset. No defaults
        var_name (str, optional): The variable name for the smoothed chlorophyll dataset. Defaults to 'smoothed_CHL'
        event_distance (int, optional): The number of days peaks must be apart to be considered separate events. Defaults to 21
        peak_window (int, optional): The number of days the chlorophyll concentration must remain above or below the threshold. Defaults to 10

    Returns: 
        List: List of bloom event and peaks within each event.
    """
    # STEP 1: Load in chlorohyll peak list
    chl_peaks = [int(p) for p in chl_peaks_list]
    if len(chl_peaks) == 0: #Returns empty list if no peaks were found
        return []
    thld_val = (float(clipped_thld.values) if isinstance(clipped_thld, xr.DataArray) else float(clipped_thld))

    # STEP 2: Defines the smoothed dataset
    if isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if var_name in dataset:
            smoothed_CHL = np.asarray(dataset[var_name]).squeeze()
        else:
            raise ValueError("Must specify correct variable name")
    elif isinstance(dataset, (xr.DataArray, pd.Series)):
        smoothed_CHL = dataset.values.squeeze()
    else:
        smoothed_CHL = np.squeeze(np.asarray(dataset))
    bloom_events = []

    # STEP 3: Creates the range for chlorophyll values to be observed in and identifies peak timeline
    current_event = [chl_peaks[0]] #Current event starts at the first peak identified
    days_between_events = event_distance #The number of days that must pass between conditions for the peaks to be considered separate events
    for i in range(1,len(chl_peaks)):
        previous_peak = int(chl_peaks[i-1]) #Finds the previous peak
        current_peak = int(chl_peaks[i])
        chl_between_peaks = smoothed_CHL[previous_peak:current_peak] #Creates a list of all chlorophyll values between the current peak and previous peak
        dropped_below_thld = False

    # STEP 4: Find if the chlorophyll concentration drops below the threshold for a certain number of consecutive days
        #Checks to see if the number of days between chlorophyll peaks is above the specified peak window
        if len(chl_between_peaks)>=peak_window:
            chl_series = pd.Series(chl_between_peaks)
            #If all chlorophyll values are below the pre-determined threshold, dropped_below_thld is true. It adds up the trues and falses and finds the spots where the value is equal to peak_window
            dropped_below_thld = (chl_series<thld_val).rolling(window=peak_window).sum().eq(peak_window).any()
    # STEP 5: Append events to events list. 
        if current_peak-previous_peak<days_between_events or not dropped_below_thld: #If peaks are too close together or does not drops below threshold, they are the same event.
            current_event.append(current_peak)
        else: #Peaks are an appropriate distance apart or chl drop below the threshold.
            bloom_events.append(current_event)
            current_event = [current_peak]
    bloom_events.append(current_event)
    return bloom_events


In [ ]:
def max_peak(event, dataset, time_array, var_name_smooth='smoothed_CHL'):
    """
    Finds the peak in an event with the maximum chlorophyll concentration for the event.
    For use in bloom_timing function.

    Args:
        event (list, required): Pre-calculated list of peaks for the event. No defaults
        dataset (xarray.Dataset, required): The dataset for analysis. No defaults
        var_name_smooth (str, optional): The variable name for the smoothed chlorophyll data. Defaults to 'smoothed_CHL'.

    Returns:
        List. A list of peaks associated with that blooms maximum chlorophyll concentration.
    """
    if hasattr(dataset, var_name_smooth):
        region_smoothed = dataset[var_name_smooth].values
    elif hasattr(dataset, 'values'):
        region_smoothed = dataset.values
    else:
        region_smoothed = np.asarray(dataset)
    region_time_smoothed = time_array #Extracts time values
    peak_chl_values = -float('inf')
    max_chl_day = None
    flatten_event = [] #For events that are multimodal, this flattens it into one list and not a tuple
    for item in event:
        if isinstance(item,(tuple,list,np.ndarray)): #If the event has more than one peak, it makes it one list and not a variety of data types.
            flatten_event.extend(item)
        else:
            flatten_event.append(item)
    for day in flatten_event:
        chl_peak = region_smoothed[int(day)] #Finds the chl value at the peak
        if chl_peak>peak_chl_values: #Checks to see if the current chl value is greater than the previous peak's value.
            peak_chl_values = chl_peak #If it is, it becomes the new maximum of the event
            max_chl_day = day #This is the day of the maximum
    peak_DOY = region_time_smoothed[max_chl_day]
    if isinstance(region_smoothed, np.ndarray):
        peak_chl = region_smoothed[max_chl_day]
    else:
        peak_chl = region_smoothed.values[max_chl_day]
    ts = pd.Timestamp(peak_DOY)
    peak_date = ts.date()
    peak_doy = ts.dayofyear
    return peak_date,peak_doy,peak_chl

In [ ]:
def max_roc_for_bloom(start_DOY,end_DOY,dataset,roc_var_name='ROC'):
    """
    Finds the maximum rate of change for each bloom.

    This function uses a pre-saved dataset of daily rates of change and pre-calculated start and end DOYs for each event
    It then finds the maximum rate of change between the initiation and termination date and then adds it to the start day value to get the DOY value for the maximum rate of change.
    This function is part of the bloom_timing function.

    Args:
        start_DOY (int, required): Pre-calculated start DOY for the event. No defaults
        end_DOY (int, required): Pre-calculated end DOY for the event. No defaults
        dataset (xarray.Dataset, required): Dataset of interest. No defaults
        roc_var_name (str, optional): Variable name for the rate of change of daily chlorophyll. Defaults to 'ROC'
    Returns:
        int/float: The maximum rates of change for the bloom.
    """
    roc = dataset[roc_var_name]
    range_roc = roc[start_DOY:end_DOY+1]
    if len(range_roc)>0:
        range_max_roc = np.nanargmax(range_roc) #Finds local maximum rate of change for each detected bloom
        max_roc = start_DOY+range_max_roc #Gets the actual day of year value
    max_roc = max_roc
    return max_roc

In [ ]:
def bloom_timing(clipped_thld,clipped_med,dataset,data_type='daily',var_name_smooth='smoothed_CHL',roc_var_name='ROC',init_term_window=5,trough_length=3,search_window=90,**kwargs):
    """
    This function finds the initiation and termination dates of blooms based on a rolling peak window.

    This function finds a variety of bloom timing values, including:
        - Bloom events: The events (and peaks within the events) throughout the dataset that satisfy the peak conditions.
        - Start day: The day since the start of the dataset where the initiation conditions were met for each bloom event.
        - End day: The day since the start of the dataset where the termination conditions were met for each bloom event
        - Peak date: The date of the maximum chlorophyll peak for each bloom event.
        - Peak DOY: The DOY (1-366) of the maximum chlorophyll peak for each bloom event.
        - Maximum chlorophyll: The maximum chlorophyll value for each bloom event.
        - Maximum rate of change: The maximum rate of change for each bloom event.
        - First exceedance day: The day for each bloom event where the chlorophyll concentration first exceed the pre-determined threshold.
        - Last dip day: The day for each bloom event where the chlorophyll concentration last dipped below the pre-determined threshold.


    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults
        dataset (xarray.Dataset, required): The daily chlorophyll dataset. No defaults
        data_type (str, optional): The type of data being used. Accepts 'daily' or 'climatology'. Defautls to 'daily'
        var_name_smooth (str, optional): The variable name for the smoothed chlorophyll data in the dataset. Defaults to 'smoothed_CHL'
        roc_var_name (str, optional): The variable name for the daily rate of change. Defaults to 'ROC'
        init_term_window (int, optional): Amount of time each condition must be met for it to trigger an initiation or termination date. Defaults to 5
        trough_length (int, optional): The number of days on either side of a minimum chl value that the value must remain below for it to be considered a trough. Defaults to 3
        search_window (int, optional): The number of days after the final peak of an event that the function searches through to find the termination date. Defaults to 90
        **kwargs: Additional input for bloom_event_detection and threshold_value functions. Possible inputs include: 
            - peak_window (int, optional): The amount of time a peak must remain above the threshold for it to be considered an event. Defaults to 10  
            - days (int, optional): Distance for find_peaks function. Defaults to 10
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.01

    Returns:
        tuple: A tuple containing (start_DOY, end_DOY, merge_bloom_events, peak_dates, peak_DOYs, maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list), where:
            start_DOY (list): The list of bloom initiation days.
            end_DOY (list): The list of bloom termination days.
            merge_bloom_events (list): The list of bloom events.
            peak_dates (list): The list of chlorophyll maximum days for each bloom event.
            peak_DOYs (list): The list of dates of the chlorophyll maximums for each bloom event.
            maximum_chl (list): The list of maximum chlorophyll values for each bloom event.
            maximum_roc (list): The list of maximum rates of change for each bloom event.
            start_at_thld_list (list): The list of DOYs where the chl first crosses the threshold for an event.
            end_at_thld_list (list): The list of DOYs where the chl last dipped below the threshold for an event.
    """
    # STEP 1: Identify chlorophyll median dataset and find the rate of change, bloom peaks, and bloom events.
    if isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if var_name_smooth in dataset:
            chl_median = np.asarray(dataset[var_name_smooth]).squeeze()
        else:
            raise ValueError("Must specify correct variable name")
    elif isinstance(dataset, (xr.DataArray, pd.Series)):
        chl_median = np.asarray(dataset).squeeze()
    else:
        chl_median = np.asarray(dataset).squeeze()

    if data_type == 'daily' and isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if roc_var_name in dataset:
            roc = np.asarray(dataset[roc_var_name]).squeeze()
        else:
            roc = np.gradient(chl_median)
    elif data_type == 'climatology' and isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if roc_var_name in dataset:
            roc = np.asarray(dataset[roc_var_name]).squeeze()
        else:
            roc = np.gradient(chl_median)
    chl_median_series = pd.Series(chl_median)
    chl_peaks_list = bloom_peak_detection(clipped_thld=clipped_thld,data=dataset,var_name=var_name_smooth,**kwargs)
    bloom_events = bloom_event_detection(dataset=dataset,clipped_thld=clipped_thld,chl_peaks_list=chl_peaks_list,var_name=var_name_smooth,**kwargs)
    
    # STEP 2: Define peak windows
    last_end_day, last_start_day = 0, 0
    start_DOY, end_DOY, merge_bloom_events = [], [], []
    peak_dates, peak_DOYs, maximum_chl = [], [], []
    maximum_roc = []
    max_index = len(roc)-1

    # STEP 3: Identify all possible initiation and termination dates for the full time series
    is_roc_negative =  roc < 0
    is_roc_positive = roc >= 0
    is_below_threshold = chl_median < clipped_thld
    is_below_median = chl_median <= clipped_med

    #Find all days for time series where initiation conditions are met (positive growth and below the threshold)
    initiation_conditions_met = pd.Series(is_roc_positive & is_below_threshold)
    initiation_rolling = initiation_conditions_met.rolling(window=init_term_window).sum()
    initiation_days = initiation_rolling[initiation_rolling == init_term_window].index.to_numpy()

    #Find all days for time series where termination conditions are met
    termination_conditions_met = pd.Series(is_roc_negative & is_below_threshold)
    termination_rolling = termination_conditions_met.rolling(window=init_term_window).sum()
    termination_days = termination_rolling[termination_rolling == init_term_window].index.to_numpy()

    #Find all local troughs for the full dataset. The troughs must have chl values lower than specified consecutive days on either side. 
    #This smooths out some of the smaller bumps caused by the noisy chl-a data and keeps major troughs. Tested 1,2,3. 
    chl_median_series = pd.Series(chl_median)
    local_minimum = (chl_median_series < chl_median_series.shift(trough_length)) & (chl_median_series < chl_median_series.shift(-trough_length)) & is_below_median
    local_minimum = local_minimum[local_minimum].index.to_numpy()

    # STEP 4: Find the initiation date of the bloom based on the rate of change.
    for event in bloom_events:
        event_start = event[0]
        event_end = event[-1]

        #Sets the end of the window to be 90 days from the last peak in the event or the end of the dataset, whichever comes first.
        end_of_window = min(max_index,event[-1]+search_window) 
        start_day = last_end_day #Sets the start of the window to the last end day

        #Find all possible initiation days between the end of the last bloom and the first peak in the current event.
        possible_init_dates = initiation_days[(initiation_days < event_start)&(initiation_days >= start_day)] 

        if len(possible_init_dates) > 0 and last_end_day >= last_start_day: #Ensures that the initiation date is not before the previous bloom's termination date
            possible_start_day = possible_init_dates[-1] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= last_end_day) & (local_minimum <= event_start)] #Finds all troughs between first peak and the previous termination

            #Attach it to the closest trough if one is available
            if len(window_troughs) > 0:
                 closest_trough = np.abs(window_troughs - possible_start_day).argmin() #Finds the closest trough to the first peak
                 start_day = window_troughs[closest_trough] 
            else:
                 start_day = possible_start_day  

    # STEP 5: Find the termination date 
        end_day = event_end
        possible_term_dates = termination_days[(termination_days>event_end) & (termination_days<=end_of_window)]

        if len(possible_term_dates)>0:
            possible_end_day = possible_term_dates[0] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
            if len(window_troughs) > 0:
                closest_trough = np.abs(window_troughs - possible_end_day).argmin() #Finds the closest trough to the first peak
                end_day = window_troughs[closest_trough]

        else: #If termination conditions are not met for a bloom, find the next trough that is below the threshold value and make that the termination date.
                potential_trough = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
                if len(potential_trough) > 0:
                    end_day = potential_trough[0]

    # STEP 6: Find peak DOY, date, and chlorophyll values for each bloom event.
        peak_date, peak_DOY, max_chl = max_peak(event,dataset,var_name_smooth=var_name_smooth)

    # STEP 7: Find the maximum rate of change for the bloom event
        max_roc = max_roc_for_bloom(start_DOY=start_day,end_DOY=end_day,dataset=dataset,roc_var_name=roc_var_name)

    # STEP 8: Merge and/or append events to the lists
        same_bloom = len(start_DOY) > 0 and start_day == start_DOY[-1] and end_day == end_DOY[-1]
        same_timing = (last_end_day>0) and (event[0]<=last_end_day)
        if same_bloom or same_timing: #This ensures that termination dates are not duplicated and every initiation date has a termination date
            merge_bloom_events[-1].extend(event)
            if end_day>end_DOY[-1]:
                end_DOY[-1] = end_day
                if max_chl > maximum_chl[-1]:
                    maximum_chl[-1] = max_chl
                    peak_dates[-1] = peak_date
                    peak_DOYs[-1] = peak_DOY
                #Recalculate the maximum ROC for the expanded timeline
                new_max_roc = max_roc_for_bloom(start_DOY=start_DOY[-1],end_DOY=end_DOY[-1],dataset=dataset)
                maximum_roc[-1] = new_max_roc

        else:
            #Documented as an entirely new event
            start_DOY.append(start_day)
            end_DOY.append(end_day)
            merge_bloom_events.append(list(event))
            peak_dates.append(peak_date)
            peak_DOYs.append(peak_DOY)
            maximum_chl.append(max_chl)
            maximum_roc.append(max_roc)
        
        last_start_day = start_day #Resets start and end dates for the loop
        last_end_day = end_day

    # STEP 9: Find the days where the chl first exceeds the threshold and where it last dips below the threshold
    chl_median_values = chl_median
    start_at_thld_list = []
    end_at_thld_list = []
    for i in range(len(start_DOY)):
        start = start_DOY[i]
        peak_start = merge_bloom_events[i][0]
        #Look forward for the day where it first crosses above the threshold
        start_found = False
        for j in range(start + 1, peak_start + 1, 1):
            if chl_median_values[j] >= clipped_thld:
                start_at_thld_list.append(j)
                start_found = True
                break
        if not start_found:
            start_at_thld_list.append(peak_start)

        end = end_DOY[i]
        peak_end = merge_bloom_events[i][-1]
        #Look backwards for the last drop below the threshold
        end_found = False
        if end > peak_end:
            for j in range (end, peak_end - 1, -1):
                if chl_median_values[j] >= clipped_thld:
                    end_at_thld_list.append(j+1)
                    end_found = True
                    break
        if not end_found:
            end_at_thld_list.append(end)
    return start_DOY,end_DOY,merge_bloom_events,peak_dates,peak_DOYs,maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list

In [ ]:
def bloom_classification_1d(dataset,bloom_events):
    """
    Classifies identified blooms by the peak DOY as spring, fall, or other.

    This function classifies a bloom as a spring bloom, fall bloom, or other bloom based on its peak DOY. Based on the seasons and relative start and peak times, the DOY ranges are
        - 1 to 60 for winter blooms (January 1 to February 28)
        - 61 to 152 for spring blooms (March 1 to June 1)
        - 153 to 243 for summer blooms (June 1 to August 31)
        - 244 to 366 for fall blooms (September 1 to December 31)
    Other blooms do not fall within these DOY ranges. 

    Args:
        dataset (xarray.Dataset, required): Dataset of region of interest. No defaults
        bloom_events (list, required): A pre-calculated list of all bloom events for the dataset

    Returns:
        Tuple: A tuple containing all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms, where:
            all_blooms: A list of all blooms as their string classification "Spring", "Fall", or "Other".
            spring_blooms: A list of all spring blooms for the dataset. Returns the peak DOY. 
            fall_blooms: A list of all fall blooms for the dataset. Returns the peak DOY.
            winter_blooms: A list of all winter blooms for the dataset. Returns the peak DOY
            other_blooms: A list of blooms not classified as spring or fall blooms. Returns peak DOY.
    """
    region_time_smoothed = dataset['time'].values
    spring_blooms = []
    fall_blooms = []
    winter_blooms = []
    other_blooms = []
    all_blooms = []
    for event in bloom_events:
        print(event)
        initial_peak = event[0]
        initial_peak_date = region_time_smoothed[initial_peak]
        initial_peak_DOY = pd.to_datetime(initial_peak_date).dayofyear
        if initial_peak_DOY >= 60 and initial_peak_DOY <=152:
            potential_spring_bloom = initial_peak_DOY
            spring_blooms.append(potential_spring_bloom)
            all_blooms.append("Spring")
        #Identify fall blooms as last bloom of the year or within the fall DOY range
        elif initial_peak_DOY >= 245 and initial_peak_DOY <=366:
            potential_fall_bloom = initial_peak_DOY
            fall_blooms.append(potential_fall_bloom)
            all_blooms.append("Fall")
        #Identify other blooms as other
        elif initial_peak_DOY >=1 and initial_peak_DOY <=59:
            potential_winter_bloom = initial_peak_DOY
            winter_blooms.append(potential_winter_bloom)
            all_blooms.append("Winter")
        else:
            potential_other_bloom = initial_peak_DOY
            other_blooms.append(potential_other_bloom)
            all_blooms.append("Summer")
    return all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms

## Part 2: Quantify Bloom Metrics

#### Part 2a: Quantify the number of bloom days per year and above the threshold

This part calculates the number of bloom days per year, per month, and the percentage of bloom days per year above the threshold. The first function, bloom_days_per_year, finds the total number of bloom days per year based on initiation and termination dates defined by the rate of change and then finds the number of bloom days above the pre-determined threshold. This can be done for a calendar (Jan - Dec) year or biological (Jul - Jun) year. The number of bloom days per month was originally used for an integrated chlorophyll heatmap but is not currently included in the bloom metrics. The percentage of bloom days above the threshold was used to track if there are any long-term trends in the time spent above the threshold.

In [1]:
def bloom_days_per_year(dataset, year, start_DOY, end_DOY, thld_start, thld_end, type='calendar'):
    """
    Calculates the number of bloom days per year from bloom initiation to termination and that are above the threshold.

    This function uses the start_DOY and end_DOY lists from the bloom_timing function.
    Then it slices the data and calculates all of the bloom days for the specified year.
    This function assumes a start date of January 1 if the start date falls in the previous year. 
    It assumes a termination date of December 31 if the termination date is in the following year.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function.
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function.
        thld_start (list, required): The list of DOY where the chlorophyll first exceed the threshold.
        thld_end (list, required): The list of DOY where the chlorophyll last dipped below the threshold.
        type (str, optional): The year type for calculation. Accepts 'calendar' (Jan - Dec) and 'biological' (July - June). Defaults to 'calendar'.

    Returns:
        Int. The number of bloom days for the given year.
    """
    if type == "calendar":
        start_time = pd.to_datetime(f"{year}-01-01")
        end_time = pd.to_datetime(f"{year}-12-31")
    elif type == "biological":
        start_time = pd.to_datetime(f"{year}-07-01")
        end_time = pd.to_datetime(f"{year+1}-06-30")
    else:
        raise ValueError("Invalid year type. Choose 'calendar' or 'biological'.")
    # STEP 1: Total number of bloom days from initiation to termination
    number_bloom_days = []
    for x in range(len(start_DOY)):
        bloom_start_date = int(start_DOY[x])
        start_date = pd.to_datetime(dataset['time'].values[bloom_start_date])
        if x < len(end_DOY):
            bloom_end_date = int(end_DOY[x])
            end_date = pd.to_datetime(dataset['time'].values[bloom_end_date])
        else:
            end_date = end_time
        overlap_start = max(start_date, start_time)
        overlap_end = min(end_date, end_time)

        if overlap_start <= overlap_end:
            amount_bloom_days = (overlap_end - overlap_start).days+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_year=sum(number_bloom_days)

    # STEP 2: The number of days per year above the threshold during a bloom
    number_days_above_thld = []
    for x in range(len(thld_start)):
        thld_bloom_start_date = int(thld_start[x])
        thld_start_date = pd.to_datetime(dataset['time'].values[thld_bloom_start_date])
        if x < len(thld_end):
            thld_bloom_end_date = int(thld_end[x])
            thld_end_date = pd.to_datetime(dataset['time'].values[thld_bloom_end_date])
        else:
            thld_end_date = end_time
        thld_overlap_start = max(thld_start_date, start_time)
        thld_overlap_end = min(thld_end_date, end_time)

        if thld_overlap_start <= thld_overlap_end:
            thld_amount_bloom_days = (thld_overlap_end - thld_overlap_start).days+1
            number_days_above_thld.append(thld_amount_bloom_days)

    annual_days_above_thld = sum(number_days_above_thld)
    return bloom_days_per_year, annual_days_above_thld

In [ ]:
def bloom_days_per_month(dataset, year, month, start_DOY, end_DOY):
    """
    Calculates the number of bloom days per month from bloom initiation to termination.

    This function uses the rolling_peak_window() function. It finds the initiation and termination for each date in the time series.
    Then it slices the data and calculates all of the bloom days for the specified month.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        month (int, required): The number of the month of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function. Defaults to end_DOy

    Returns:
        Int. The number of bloom days for the given year.
    """
    number_bloom_days = []
    _, last_day_month = calendar.monthrange(year,month)
    month_start = pd.Timestamp(year=year,month=month,day=1)
    month_end = pd.Timestamp(year=year, month=month,day=last_day_month)

    for x in range(len(start_DOY)):
        start_index = int(start_DOY[x])
        bloom_start_date = pd.to_datetime(dataset['time'].values[start_index])
        if x < len(end_DOY):
            end_index = int(end_DOY[x])
            bloom_end_date = pd.to_datetime(dataset['time'].values[end_index])
        else:
            bloom_end_date = pd.Timestamp(year=year,month=12,day=31)
        start_overlap = max(bloom_start_date,month_start)
        end_overlap = min(bloom_end_date,month_end)

        if start_overlap <= end_overlap:
            amount_bloom_days = (end_overlap-start_overlap).days+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_month=sum(number_bloom_days)
    return bloom_days_per_month

In [ ]:
def percent_bloom_days(dataset, year, start_DOY, end_DOY, thld_start, thld_end):
    """
    Calculates the percent of the number of bloom days that are above the threshold for a given year.

    This function uses the bloom_days_per_year() function to calculate the total number of bloom days and the number of days above the threshold.
    It is a helper function for the larger bloom_metrics function.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        start_DOY (list, required): A pre-calculated list of start DOY indices. No defaults
        end_DOY (list, required): A pre-calculated list of end DOY indices. No defaults
        thld_start (list, required): A pre-calculated list of indices where the chlorophyll first exceeds the threshold. No defaults
        thld_end (list, required): A pre-calculated list of indices where the chlorophyll last dips below the threshold. No defaults

    Returns:
        Float. The percent of bloom days that are above the threshold for the given year.
    """
    annual_bloom_days, annual_days_above_thld = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
    bloom_days_total = annual_bloom_days
    days_above_thld = annual_days_above_thld
    percent = days_above_thld/bloom_days_total
    return percent

#### Part 2b: Quantify the Number of Phytoplankton Blooms Per Year

In [ ]:
def annual_events(peak_date,first_year=1998,last_year=2026,type='calendar'):
    """
    Finds the number of blooms per year for the dataset.

    This function categorizes blooms into years based on their peak date. It uses the max_peaks function.

    Args:
        peak_date (list, required): List of peak chl dates for the dataset. No defaults.
        first_year (int, optional): First year in the time series of interest. Defaults to 1998.
        last_year (int, optional): Year after the last year of interest in the time series. Defaults to 2026

    Returns:
        Dictionary: A dictionary of the year and the number of events in that year.
    """
    #Year
    peak_years = []
    for date in peak_date:
        dt = pd.to_datetime(date)
        if type == 'calendar':
            peak_years.append(dt.year)
        elif type == 'biological':
            bio_year = dt.year if dt.month >= 7 else dt.year - 1
            peak_years.append(bio_year)
        else: 
            raise ValueError("Invalid year type. Choose 'calendar' or 'biological'.")
        
    blooms_per_year = Counter(peak_years)
    blooms_per_year = {year: blooms_per_year.get(year,0) for year in range (int(first_year),int(last_year))}
    return blooms_per_year

#### Part 2c: Bloom Durations

In [ ]:
def bloom_duration(bloom_index,start_DOY,end_DOY,dataset,thld_start,thld_end,var_name_smooth='smoothed_CHL',data_type='daily'):
    """
    Finds the length of each bloom event.

    Given the bloom index (1 to the length of start_DOY), the start DOY is subtracted from the end DOY to get the total duration of the bloom.
    It does this for the total bloom duration and the duration above the threshold.

    Args: 
        bloom_index : Bloom index of interest. Values range from 1 to len(start_DOY) + 1. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        thld_start (list, required): Pre-calculated DOY list of days where the chl first crosses the threshold.
        thld_end (list, required): Pre-calculated DOY list of days where the chl last dipped below the threshold
        var_name_smooth (str, optional): The name of the smoothed chlorophyll dataset. Defaults to 'smoothed_CHL'
        data_type (str, optional): The type of data being processed. Defaults to 'daily'

    Returns:
        tuple. A tuple of integers for the number of days of the bloom and the number of days above the threshold.
    """
    if isinstance(dataset, (xr.Dataset, pd.DataFrame)):
        if var_name_smooth in dataset:
            chl_median = dataset[var_name_smooth]
        else:
            raise ValueError("Must specify correct variable name")
    elif isinstance(dataset, (xr.DataArray, pd.Series)):
        chl_median = dataset
    else:
        chl_median = np.asarray(dataset)
    if data_type == 'climatology':
        chl_median = chl_median
    else:
        if isinstance(chl_median, xr.DataArray):
            chl_median = chl_median.interpolate_na(dim='time',method='linear')
            chl_median = chl_median.to_numpy().squeeze()
        elif isinstance(chl_median, (pd.Series, pd.DataFrame)):
            chl_median = chl_median.interpolate(method='linear')
            chl_median = chl_median.to_numpy().squeeze()
        else:
            data = chl_median.squeeze()
            nans = np.isnan(data)
            if np.any(nans):
                x = lambda z: z.nonzero()[0]
                data[nans] = np.interp(x(nans), x(~nans), data[~nans])
            chl_median = data

    # STEP 1: Find the duration of the total bloom
    start_date = start_DOY[bloom_index]
    end_date = end_DOY[bloom_index]
    duration = end_date-start_date

    # STEP 2: Find the duration above the threshold for the bloom
    if thld_start[bloom_index] is not None:
        if bloom_index < len(thld_end):
            thld_end_idx = thld_end[bloom_index]
        else:
            thld_end_idx = len(chl_median)-1
        bloom_duration_thld = thld_end_idx-thld_start[bloom_index]
    else:
        bloom_duration_thld = "N/A"

    return duration, bloom_duration_thld

In [ ]:
def threshold_base_differences(bloom_index, start_DOY_list=None, end_DOY_list=None, thld_start_list=None, thld_end_list=None, type='both'):
    """
    Calculates the days between the ROC-based start/end DOY and the first exceedance/last drop DOY (based on threshold)

    This function uses pre-calculated lists of start DOY, end DOY, threshold start DOY, and threshold end DOY to calculate the number of days between the points.
    This is used for determining the relationship between the ROC based metrics and the threshold based metrics.
    This function can do this for just initiation DOY, just termination DOY, or both.

    Args:
        bloom_index (int, required): The index of the bloom of interest from the start_DOY list. No defaults
        start_DOY_list (list, optional): The list of start DOYs based on ROC. Must provide list for type == 'both' or 'start'. Defaults to None
        end_DOY_list (list, optional): The list of end DOYs based on ROC. Must provide list for type == 'both' or 'end'. Defaults to None
        thld_start_list (list, optional): The list of start DOYs based on the threshold. Must provide list for type == 'both' or 'start'. Defaults to None
        thld_end_list (list, optional): The list of end DOYs based on the threshold. Must provide list for type == 'both' or 'end'. Defaults to None
        type (str, optional): The metric you would like to work with. Can be 'both', 'start', or 'end'. Defaults to 'both'
    
    Returns:
        tuple. A tuple of integers including (abs_distance_start, abs_distance_end), where:
            - abs_distance_start (int): The absolute value of the number of days between the initiation DOY and the first exceedance DOY.
            - abs_distance_end (int): The absolute value of the number of days between the last drop below the threshold and the termination DOY.

    """
    if type == 'both':
        if start_DOY_list is None:
            raise ValueError("Must specify start DOY list.")
        elif end_DOY_list is None:
            raise ValueError("Must specify end DOY list")
        elif thld_start_list is None:
            raise ValueError("Must specify threshold start DOY list")
        elif thld_end_list is None:
            raise ValueError("Must specify threshold end DOY list")
        distance = thld_start_list[bloom_index] - start_DOY_list[bloom_index]
        abs_distance_start = abs(distance)
        end_distance = thld_end_list[bloom_index] - end_DOY_list[bloom_index]
        abs_distance_end = abs(end_distance)
    elif type == 'start':
        if start_DOY_list is None:
            raise ValueError("Must specify start DOY list.")
        elif thld_start_list is None:
            raise ValueError("Must specify threshold start DOY list")
        distance = thld_start_list[bloom_index] - start_DOY_list[bloom_index]
        abs_distance_start = abs(distance)
        if thld_start_list[bloom_index] < start_DOY_list[bloom_index]:
            abs_distance_start = 365 - abs_distance_start
        abs_distance_end = None
    elif type == 'end':
        if end_DOY_list is None:
            raise ValueError("Must specify end DOY list")
        elif thld_end_list is None:
            raise ValueError("Must specify threshold end DOY list")
        end_distance = thld_end_list[bloom_index] - end_DOY_list[bloom_index]
        abs_distance_end = abs(end_distance)
        if thld_end_list[bloom_index] < end_DOY_list[bloom_index]:
            abs_distance_end = 365 - abs_distance_end
        abs_distance_start = None
    else:
        raise ValueError("Must choose 'both', 'start', or 'end'.")
    return abs_distance_start, abs_distance_end


#### Part 2d: Integrated chlorophyll a

In [ ]:
def event_integrated_chla(raw_time,raw_chl,bloom_index,start_DOY,end_DOY,clipped_thld=None,type='total'):
    """
    Calculates the integrated chlorophyll-a concentration per bloom.

    This function uses trapezoid integration to estimate the amount of chlorophyll-a per bloom.
    It includes all chl-a values (0 to maximum value for the peak).
    The bounds of integration are determined by the start and end date found from the bloom_timing function.

    Args:
        raw_time (numpy.ndarray): The array of the raw time values for the dataset. No defaults.
        raw_chl (numpy.ndarray): The array of the raw chl values for the dataset. No defaults
        bloom_index (int, required): The bloom number for that index. In range of 0 - len(start_DOY). No defaults 
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. No defaults
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. No defaults
        clipped_thld (float, optional): The threshold value for the dataset. Required if type is 'threshold'. Defaults to None.
        type (str, optional): The type of integration to perform. Can be 'total' for all chl values or 'threshold' for only chl values above the threshold. Defaults to 'total'.
    
    Returns:
        Float. Integrated chlorophyll-a value for the bloom. 
    """
    bloom_start_time = np.asarray(start_DOY[bloom_index]).flatten().astype(int)
    bloom_end_time = np.asarray(end_DOY[bloom_index]).flatten().astype(int)

    lower_bound = int(bloom_start_time[0])
    upper_bound = int(bloom_end_time[0])
    bounded_time = raw_time[lower_bound:upper_bound]
    bounded_time = (bounded_time - bounded_time[0])/np.timedelta64(1,'D')
    bounded_chl = raw_chl[lower_bound:upper_bound]
    if type == 'total':
        integrated_chl = scipy.integrate.trapezoid(bounded_chl,bounded_time,axis=0)
    elif type == 'threshold':
        if clipped_thld is None:
            raise ValueError("Must specify threshold value")
        chl_above_thld = np.where(bounded_chl >= clipped_thld, bounded_chl, 0)
        integrated_chl = scipy.integrate.trapezoid(chl_above_thld,bounded_time,axis=0)
    else:
        raise ValueError("Must pick valid type: 'total' or 'threshold'.")
    return integrated_chl

In [ ]:
def annual_integrated_chl(raw_time,raw_chl,year,start_DOY,end_DOY,peak_dates,type="calendar"):
    """
    Calculates the integrated chlorophyll for a year, considering only the chlorophyll during bloom events and the whole year.

    This function first calculates the total integrated chlorophyll for the whole year.
    Then it isolates the yearly chlorophyll into only the chlorophyll during bloom events.
    It then integrates over the full year to get the total integrated chlorophyll in mg/m^3 * days for only bloom periods.
    You can choose if the year runs from Jan 1 - Dec 31 (calendar) or July 1 to June 30 (biological).

    Args:
        raw_time (numpy.ndarray): The array of the raw time values for the dataset. No defaults.
        raw_chl (numpy.ndarray): The array of the raw chl values for the dataset. No defaults
        year (int, required): Year of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated. No defaults
        end_DOY (list, required): List of end DOYs pre-calculated. No defaults
        peak_dates (list, required): List of peak dates pre=calculated. No defaults
        type (str, optional): Sets up the type of year you are integrating over. Accepts "calendar" (Jan - Dec), "biological" (July - June), "bloom" (start of first bloom to end of last bloom for the year), or "bloom_bio" (biological bloom year). Defaults to "calendar"

    Returns:
        tuple. A tuple containing two float values where the first is the total integrated chlorophyll for the year and the second is only considering the bloom periods.
    """
    if type == "calendar":
        start_time = np.datetime64(f"{year}-01-01")
        end_time = np.datetime64(f"{year}-12-31")
    elif type == "biological":
        start_time = np.datetime64(f"{year}-07-01")
        end_time = np.datetime64(f"{year+1}-06-30")
    elif type == "bloom" or type == 'bloom_bio':
        # Identify all blooms that initiated in this bloom year to find the dynamic start dates
        bloom_times = []
        for s_idx, e_idx, p_date in zip(start_DOY, end_DOY, peak_dates):
            b_start = int(np.ravel(s_idx)[0])
            b_end = int(np.ravel(e_idx)[0]) + 1
            b_time = raw_time[b_start:b_end]
            peak_time = np.datetime64(np.ravel(p_date)[0])
            peak_dt = pd.to_datetime(peak_time)
            if type == 'bloom_bio':
                b_year = peak_dt.year if peak_dt.month >= 7 else peak_dt.year - 1
            else:
                b_year = peak_dt.year
            if b_year == year:
                bloom_times.append(b_time)
        if not bloom_times:
            return 0.0, 0.0
        start_time = min(t[0] for t in bloom_times)
        end_time = max(t[-1] for t in bloom_times)

    else:
        raise ValueError("Invalid year type. Choose 'calendar', 'biological', 'bloom', 'bloom_bio'.")

    # STEP 1: Total yearly chlorophyll
    mask = (raw_time >= start_time) & (raw_time <= end_time)
    time_year = raw_time[mask]
    chl_year = raw_chl[mask]

    valid_mask = ~np.isnan(chl_year)
    time_year = time_year[valid_mask]
    chl_year = chl_year[valid_mask]

    if len(chl_year) < 2:
        annual_integrated_chl = 0.0
    else:
        bounded_time = (time_year - start_time)/np.timedelta64(1,'D')
        annual_integrated_chl = scipy.integrate.trapezoid(chl_year,bounded_time,axis=0)

    # STEP 2: Yearly integrated chlorophyll only including times of bloom 
    total_yearly_chl_during_bloom = 0.0
    for s_idx, e_idx, p_date in zip(start_DOY, end_DOY, peak_dates):
        bloom_start_time = int(np.ravel(s_idx)[0])
        bloom_end_time = int(np.ravel(e_idx)[0])+1

        bloom_time = raw_time[bloom_start_time:bloom_end_time]
        bloom_chl = raw_chl[bloom_start_time:bloom_end_time]
        peak_time = np.datetime64(np.ravel(p_date)[0])
        peak_dt = pd.to_datetime(peak_time)
        
        if type == "bloom" or type == 'bloom_bio':
            if type == 'bloom_bio':
                bloom_year = peak_dt.year if peak_dt.month >= 7 else peak_dt.year - 1
            else:
                bloom_year = peak_dt.year
            if bloom_year != year:
                continue
            bloom_bounded_time = bloom_time
            bloom_bounded_chl = bloom_chl
        else:
            year_mask = (bloom_time >= start_time)&(bloom_time<=end_time)
            if not year_mask.any():
                continue
        
            bloom_bounded_time = bloom_time[year_mask]
            bloom_bounded_chl = bloom_chl[year_mask]
        
        if len(bloom_bounded_time)>1:
            bloom_bounded_time = (bloom_bounded_time - bloom_bounded_time[0])/np.timedelta64(1,'D')
            bloom_integrated_chl = scipy.integrate.trapezoid(bloom_bounded_chl,bloom_bounded_time,axis=0)
            if not np.isnan(bloom_integrated_chl):
                total_yearly_chl_during_bloom += bloom_integrated_chl

    return annual_integrated_chl, total_yearly_chl_during_bloom

In [ ]:
def monthly_integrated_chl(dataset,year,month):
    """
    Calculates the monthly integrated chl-a concentration (mg/m^3 * day)

    This function finds the total integrated chl-a concentration from the first day to the last day of the month of the year in question.

    Args:
        dataset (xarray.Dataset, required): Raw dataset (not smoothed) for analysis. No defaults
        year (int, required): The year for calculating the integrated chlorophyll. No defaults
        month (int, required): The month number from 1-12.

    Returns:
        Float. The total integrated chlorophyll for the year as one number. 
    """
    month_string = f"{month:02d}"
    time_slice = f"{year}-{month_string}"
    raw_data = dataset.sel(time=time_slice)
    if raw_data['time'].size <= 1:
        return 0.0
    chl = raw_data['CHL_bloom_only_nan'].values
    if np.isnan(chl).all():
        return 0.0
    time = raw_data['time'].values
    start_date = np.datetime64(f"{year}-{month_string}-01")

    bounded_time = (time - start_date)/np.timedelta64(1,'D')

    valid_mask = ~np.isnan(chl)
    valid_chl = chl[valid_mask]
    valid_time = bounded_time[valid_mask]

    if len(valid_chl) <= 1:
        return 0.0

    integrated_chl = scipy.integrate.trapezoid(valid_chl,valid_time,axis=0)
    return integrated_chl

In [ ]:
def percent_annual_integrated_chl(dataset,year,bloom_index,start_DOY,end_DOY):
    """
    Calculates the percentage of the annual integrated chlorophyll that one bloom makes up.
    This function uses the yearly_integrated_chl and event_integrated_chla functions.

    Args: 
        dataset (xaray.Dataset,required): Raw dataset for analysis. No defaults
        year (int, required): The year for calculating total integrated chlorophyll. No defaults
        bloom_index (int, required): The index of the bloom for calculation. No defaults.
        start_DOY (list, required): The pre-calculated list of initiation DOYs. No defaults
        end_DOY (list, required): The pre-calculated list of termination DOYs. No defaults
    
    Returns:
        tuple. A tuple of two float values. The first is the percentge of the total annual chlorophyll and the second is the percentage of the annual chlorophyll in bloom periods.
    """
    total_annual_chl, bloom_period_integrated_chl = annual_integrated_chl(dataset,year,start_DOY=start_DOY,end_DOY=end_DOY)
    bloom_chl = event_integrated_chla(dataset,bloom_index,start_DOY=start_DOY,end_DOY=end_DOY)
    percent_of_total_annual = (bloom_chl/total_annual_chl)
    percent_bloom_period_annual = (bloom_chl/bloom_period_integrated_chl)
    return percent_of_total_annual, percent_bloom_period_annual

#### Part 2e: Individual Bloom metrics
This creates a dataframe with a variety of bloom metrics for each bloom event. 

In [ ]:
def bloom_metrics(clipped_thld,clipped_med,dataset,data_type='daily',var_name_smooth='ssmoothed_CHL',roc_var_name='ROC',verbose=False):
    """
    Finds bloom metrics for each bloom in the dataset.

    This function computes the following metrics and returns them as a dataframe with the following labels:
        - Year: The year the bloom event peaked (int)
        - Biological Year: The biological year the bloom event peaked (int)
        - Bloom_classification: The classification (spring, fall, winter, other) (str)
        - Start_date: The date the bloom began based on the rate of change (str)
        - Peak_date: The date the bloom peaked based on the maximum chlorophyll concentration (str)
        - End_date: The date the bloom ended based on the rate of change (str)
        - Start_DOY: The DOY (1-366) that the bloom initiated (int)
        - End_DOY: The DOY (1-366) that the bloom terminated (int)
        - Peak_DOY: The DOY (1-366) that the bloom peaked (int)
        - Total_duration: The total number of days that bloom lasted (int)
        - Number_of_peaks: The total number of peaks in that bloom event (int)
        - First_exceed_date: The date the bloom first exceeded the threshold (str)
        - last_drop_date: The date the bloom last dropped below the threshold (str)
        - First_exceed_DOY: The DOY (1-366) that the bloom first exceeded the threshold (int)
        - last_drop_DOY: The DOY (1-366) that the bloom last dropped below the threshold (int)
        - Days_between_initiations: The number of days between the ROC-based initiation and the first exceedance of the threshold (int)
        - Days_between_terminations: The number of days between the ROC-based termination and the last drop below the threshold (int)
        - Duration_above_threshold: The number of days a bloom spent above the threshold (int)
        - Bloom_Integrated_Chlorophyll: The total integrated chlorophyll concentration for the event (float)
        - Bloom_Integrated_Chlorophyll_Above_Threshold: The amount of integrated chlorophyll above the threshold (float)
        - Maximum_chlorophyll: The maximum chlorophyll concentration for the event (float)
        - Percent_Annual_Integrated_Chlorophyll: The percent of the annual integrated chlorophyll made up by the bloom (float)
        - Percent_Bloom_Period_Annual_Integrated_Chlorophyll: The percent of the bloom period only annual integrated chlorophyll made up by the bloom (float)
        - Percent_Bio_Year_Integrated_Chlorophyll: The percent of the biological year chlorophyll made up by the bloom (float)
        - Percent_Bio_Year_Bloom_Only_Chlorophyll: The percent of the biological year chlorophyll during bloom periods only made up by the bloom (float)

    Args:
        clipped_thld (float, required): The pre-calculated climatological threshold for the area. No defaults
        clipped_med (float, required): The pre-calculated climatological median for the area. No defaults
        dataset (xarray.Dataset): The dataset for analysis. No defaults
        data_type (str, optional): The type of data being processed. Accepts 'daily' or 'climatology'. Defaults to 'daily'
        var_name_smooth (str, optional): The name of the smoothed chlorophyll variable. Defaults to 'smoothed_CHL'
        roc_var_name (str, optional): The name of the rate of change variable. Defaults to 'ROC'
        verbose (Boolean, optional): Shows the print statements for checking if variables were successfully computed if True. Defaults to False
    Returns:
        pandas.DataFrame. A pandas dataframe of the bloom metrics for the region.
    """
    # STEP 1: Run bloom_timing function for bloom timing metrics
    start_DOY,end_DOY,bloom_events,peak_dates,peak_DOY,max_chl,_, thld_start,thld_end = bloom_timing(clipped_thld=clipped_thld,clipped_med=clipped_med,dataset=dataset,data_type=data_type,var_name_smooth=var_name_smooth,roc_var_name=roc_var_name,prm=0.02)
    if verbose is True:
        print("Successfully found bloom timing metrics")

    # STEP 2: Create a bloom ID for each event
    bloom_indices = []
    for i in range(len(start_DOY)):
        bloom_index = i+1
        bloom_indices.append(bloom_index)
    if verbose is True:
        print("Successfully created bloom indices")

    #STEP 3: Identify bloom year based on peak dates
    peak_year = [date.year for date in peak_dates]
    bio_year = []
    for date in peak_dates:
        dt = pd.to_datetime(date)
        peak_bio_year = dt.year if dt.month >= 7 else dt.year - 1
        bio_year.append(peak_bio_year)
    if verbose is True:
        print("Successfully found bloom years")

    # STEP 4: Convert DOY values from bloom_timing function into dates and 1-366 DOY values.
    start_date_str = []
    start_365_DOY = []
    end_date_str = []
    end_365_DOY = []
    for i in range(len(start_DOY)):
        start_date_with_time = pd.to_datetime(dataset['time'].values[int(start_DOY[i])])
        start_date = start_date_with_time.date()
        start_DOY_365 = start_date_with_time.dayofyear
        if i<len(end_DOY):
            end_date_with_time = pd.to_datetime(dataset['time'].values[int(end_DOY[i])])
            end_date = end_date_with_time.date()
            end_DOY_365 = end_date_with_time.dayofyear
        else:
            end_date = "N/A"
            end_DOY_365 = "N/A"
        start_date_str.append(start_date)
        start_365_DOY.append(start_DOY_365)
        end_date_str.append(end_date)
        end_365_DOY.append(end_DOY_365)
    if verbose is True:
        print("Successfully converted bloom timing outputs to dates and DOYs")
    # STEP 5: Classify blooms as winter, spring, fall, or other
    bloom_class = bloom_classification(dataset=dataset,bloom_events=bloom_events)[0]
    if verbose is True:
        print("Successfully classified blooms")

    # STEP 6: Find dates where chl first and last crossed threshold
    start_date_thld = []
    start_thld_365_DOY = []
    end_date_thld = []
    end_thld_365_DOY = []
    for i in range(len(thld_start)):
        thld_start_date_time = pd.to_datetime(dataset['time'].values[int(thld_start[i])])
        thld_start_date = thld_start_date_time.date()
        thld_start_DOY_365 = thld_start_date_time.dayofyear
        if i<len(thld_end):
            thld_end_date_with_time = pd.to_datetime(dataset['time'].values[int(thld_end[i])])
            thld_end_date = thld_end_date_with_time.date()
            thld_end_DOY_365 = thld_end_date_with_time.dayofyear
        else:
            thld_end_date = "N/A"
            thld_end_DOY_365 = "N/A"
        start_date_thld.append(thld_start_date)
        start_thld_365_DOY.append(thld_start_DOY_365)
        end_date_thld.append(thld_end_date)
        end_thld_365_DOY.append(thld_end_DOY_365)
    if verbose is True:
        print("Successfully found threshold based dates and DOYs")

    # STEP 7: Find the total duration of each bloom 
    bloom_total_duration = []
    bloom_thld_duration = []
    for i in range(len(start_DOY)):
        bloom_length, bloom_length_thld = bloom_duration(i,start_DOY=start_DOY,end_DOY=end_DOY,dataset=dataset,thld_start=thld_start,thld_end=thld_end,var_name_smooth=var_name_smooth, data_type=data_type)
        bloom_total_duration.append(bloom_length)
        bloom_thld_duration.append(bloom_length_thld)
    if verbose is True:
        print("Successfully found bloom durations")
    
    # STEP 8: Integrated chlorophyll statistics
    full_chl_array = dataset[var_name_smooth].values
    full_time_array = dataset['time'].values
    annual_cache = {}
    bio_cache = {}
    event_chl = []
    thld_chl = []
    percent_chl = []
    percent_bloom_period_annual = []
    percent_bio_chl = []
    percent_bloom_period_bio = []

    year_to_bloom_indices = defaultdict(list)
    for i, peak_date in enumerate(peak_dates):
        year_to_bloom_indices[peak_date.year].append(i)
    for year in range(1998,2026):
        annual_cache[year] = annual_integrated_chl(raw_chl=full_chl_array,raw_time=full_time_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates)
        total_annual, bloom_period_annual = annual_cache[year]
        bio_cache[year] = annual_integrated_chl(raw_chl=full_chl_array,raw_time=full_time_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type='biological')
        total_bio_annual, bloom_period_bio = bio_cache[year]
        bloom_indices = year_to_bloom_indices.get(year, [])
        if bloom_indices:
            for i in bloom_indices:
                bloom_chl = event_integrated_chla(raw_time=full_time_array,raw_chl=full_chl_array,bloom_index=i,start_DOY=start_DOY,end_DOY=end_DOY)
                percent_annual = (bloom_chl/total_annual) if total_annual > 0 else 0
                percent_bloom_period = (bloom_chl/bloom_period_annual) if bloom_period_annual > 0 else 0
                percent_chl.append(percent_annual)
                percent_bloom_period_annual.append(percent_bloom_period)
                event_chl.append(bloom_chl)
                percent_bio_annual = (bloom_chl/total_bio_annual) if total_bio_annual > 0 else 0
                percent_bio_chl.append(percent_bio_annual)
                percent_bio_bloom_annual = (bloom_chl/bloom_period_bio) if bloom_period_bio > 0 else 0
                percent_bloom_period_bio.append(percent_bio_bloom_annual)
                thld_int_chl = event_integrated_chla(raw_time=full_time_array,raw_chl=full_chl_array,bloom_index=i,start_DOY=start_DOY,end_DOY=end_DOY, clipped_thld=clipped_thld, type='threshold')
                thld_chl.append(thld_int_chl)
        else:
            event_chl.append(np.nan)
            thld_chl.append(np.nan)
            percent_chl.append(0.0)
            percent_bloom_period_annual.append(0.0)
            percent_bio_chl.append(0.0)
            percent_bloom_period_bio.append(0.0)
    if verbose is True:
        print("Successfully integrated chlorophyll")

    # STEP 10: Find the number of peaks per event
    number_of_peaks = []
    for event in bloom_events:
        length = len(event)
        number_of_peaks.append(length)
    
    # STEP 11: Find the number of days between initiation(termination) and threshold initiation(termination) days.
    days_between_initiation = []
    days_between_termination = []
    for i in range(len(start_DOY)):
        values = threshold_base_differences(bloom_index=i,start_DOY_list=start_DOY,end_DOY_list=end_DOY,thld_start_list=thld_start,thld_end_list=thld_end)
        initiation = int(values[0])
        termination = int(values[1])
        days_between_initiation.append(initiation)
        days_between_termination.append(termination)
    dataframe_data = {
        "Year": peak_year,
        "Biological Year": bio_year,
        "Bloom_classification": bloom_class,
        "Start_date": start_date_str,
        "Peak_date": peak_dates,
        "End_date": end_date_str,
        "Start_DOY": start_365_DOY,
        "End_DOY": end_365_DOY,
        "Peak_DOY": peak_DOY,
        "Total_duration": bloom_total_duration,
        "Number_of_peaks": number_of_peaks,
        "First_exceed_date": start_date_thld,
        "last_drop_date": end_date_thld,
        "First_exceed_DOY": start_thld_365_DOY,
        "last_drop_DOY": end_thld_365_DOY,
        "Duration_above_threshold": bloom_thld_duration,
        "Days_between_initiations": days_between_initiation,
        "Days_between_terminations": days_between_termination,
        "Bloom_Integrated_Chlorophyll": event_chl,
        "Bloom_Integrated_Chlorophyll_Above_Threshold": thld_chl,
        "Maximum_chlorophyll": max_chl,
        "Percent_Annual_Integrated_Chlorophyll": percent_chl,
        "Percent_Bloom_Period_Annual_Integrated_Chlorophyll": percent_bloom_period_annual,
        "Percent_Bio_Year_Integrated_Chlorophyll": percent_bio_chl,
        "Percent_Bio_Year_Bloom_Only_Chlorophyll": percent_bloom_period_bio
    }
    bloom_df = pd.DataFrame(dataframe_data,index=bloom_indices)
    bloom_df.index.name = "Bloom ID"
    return bloom_df

#### Part 2f: Regional Summary Metrics
This finds annual and monthly metrics for the region. 

In [ ]:
def summary_bloom_metrics(clipped_thld,clipped_med,dataset,verbose=False):
    """
    Finds summary (yearly and monthly) metrics for the region.

    This function finds the following metrics and returns as dataframe with the following labels:
    - Number_of_blooms: The number of blooms in that year based on the peak date (int)
    - Number_of_blooms_biological: The number of blooms in the biological year based on peak date (int)
    - Total_bloom_days: The total number of bloom days in that year (Jan 1 - Dec 31) (int)
    - Bloom_days_above_threshold: The number of days that year spent above the threshold during a bloom period (int)
    - Total_bloom_days_biological: The total number of bloom days in the biological year (int)
    - Bloom_days_above_threshold_biological: The number of days in the biological year above the threshold (int)
    - Percent_total_bloom_days_above_threshold: The percent of days above the threshold in a calendar year compared to total bloom days (float)
    - Percent_total_bloom_days_above_threshold_biological: The percent of days above the threshold in a biological year compared to total bloom days (float)
    - Total_integrated_chl: The total integrated chlorophyll for the year (Jan 1 - Dec 31) (float)
    - Bloom_period_integrated_chl: The total integrated chlorophyll for the year (Jan 1 - Dec 31) during bloom periods only (float)
    - Biological_total_integrated: The total integrated chlorophyll for the biological year (July 1 - June 30) (float)
    - Biological_bloom_period_integrated: The total integrated chlorophyll for the biological year (July 1 - June 30) during bloom periods only (float)
    - January_integrated_chl: The integrated chlorophyll for January based on bloom periods only
    - February_integrated_chl: The integrated chlorophyll for February based on bloom periods only
    - March_integrated_chl: The integrated chlorophyll for March based on bloom periods only
    - April_integrated_chl: The integrated chlorophyll for April based on bloom periods only
    - May_integrated_chl: The integrated chlorophyll for May based on bloom periods only
    - June_integrated_chl: The integrated chlorophyll for June based on bloom periods only
    - July_integrated_chl: The integrated chlorophyll for July based on bloom periods only
    - August_integrated_chl: The integrated chlorophyll for August based on bloom periods only
    - September_integrated_chl: The integrated chlorophyll for September based on bloom periods only
    - October_integrated_chl: The integrated chlorophyll for October based on bloom periods only
    - November_integrated_chl: The integrated chlorophyll for November based on bloom periods only
    - December_integrated_chl: The integrated chlorophyll for December based on bloom periods only

    Args:
        clipped_thld (float, required): The pre-calculated climatological threshold for the area. No defaults
        clipped_med (float, required): The pre-calculated climatological median for the area. No defaults
        dataset (xarray.Dataset): The dataset for analysis. No defaults
        verbose (Boolean, optional): Shows the print statements for checking if variables were successfully computed if True. Defaults to False
    Returns:
        pandas.DataFrame. A pandas dataframe of the summary bloom metrics for the region.
    """
    # STEP 1: Run bloom_timing function for bloom timing metrics
    start_DOY,end_DOY,_,peak_dates,_,_,_,thld_start,thld_end = bloom_timing(clipped_thld=clipped_thld,clipped_med=clipped_med,dataset=dataset)
    if verbose is True:
        print("Successfully found bloom timing metrics")

    # STEP 2: Find the number of events per year (and the years)
    bloom_events_annual = annual_events(peak_date=peak_dates)
    bloom_events_bio = annual_events(peak_date=peak_dates,type='biological')
    if verbose is True:
        print("Successfully calculated the number of events per year")

    # STEP 3: Find the total number of bloom days and days above the threshold per year.
    bloom_days_annual, bloom_days_annual_above_thld = [],[]
    bio_bloom_days_annual, bio_bloom_days_above_thld = [], []
    for year in range(1998,2026):
        annual, bloom_period = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
        if verbose is True:
            print(f"Year: {year} | Total Bloom Days: {annual} | Days Above Threshold: {bloom_period}")
        bloom_days_annual.append(annual)
        bloom_days_annual_above_thld.append(bloom_period)
        annual_bio, bloom_period_bio = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end,type = 'biological')
        bio_bloom_days_annual.append(annual_bio)
        bio_bloom_days_above_thld.append(bloom_period_bio)
    if verbose is True:
        print("Successfully found the annual bloom days and days above the threshold")

    # STEP 4: Find the percentage of total bloom days that are above the threshold
    percent_annual_thld = []
    for year in range(1998,2026):
        percent = percent_bloom_days(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
        percent_annual_thld.append(percent)
    percent_bio_thld = []
    for i in range(len(bio_bloom_days_annual)):
        if bio_bloom_days_annual[i] > 0:
            percent_bio = bio_bloom_days_above_thld[i]/bio_bloom_days_annual[i]
            percent_bio_thld.append(percent_bio)
        else:
            percent_bio_thld.append(0)
    if verbose is True:
        print("Successfully found the percentage of total bloom days above the threshold")

    # STEP 5: Find the annual integrated chlorophyll (total and bloom period only)
    full_chl_array = dataset['CHL_median'].values
    full_time_array = dataset['time'].values
    integrated_annual = []
    bloom_integrated_annual = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates)
        integrated_annual.append(annual_int)
        bloom_integrated_annual.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated the integrated chlorophyll for the calendar year")
    
    # STEP 6: Find the biological (July - June) year integrated chlorophyll
    biological_integrated_annual = []
    biological_bloom_int_annual = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type="biological")
        biological_integrated_annual.append(annual_int)
        biological_bloom_int_annual.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated integrated chlorophyll for biological year")

    # STEP 7: Find the bloom year (start of first bloom, end of last bloom) integrated chlorophyll
    bloom_year_integrated = []
    bloom_year_bloom_only = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type="bloom")
        bloom_year_integrated.append(annual_int)
        bloom_year_bloom_only.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated integrated chlorophyll for bloom year")

    # STEP 8: Monthly integrated chlorophyll per year (total and bloom period only)
    year = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
    month_options = ['January','February','March','April','May','June','July','August','September','October','November','December']
    jan_chl,feb_chl,mar_chl,apr_chl,may_chl,jun_chl,jul_chl,aug_chl,sep_chl,oct_chl,nov_chl,dec_chl = [],[],[],[],[],[],[],[],[],[],[],[]
    lists = [jan_chl,feb_chl,mar_chl,apr_chl,may_chl,jun_chl,jul_chl,aug_chl,sep_chl,oct_chl,nov_chl,dec_chl]
    for j in range(len(month_options)):
        month_actual = j+1
        monthly_chl = lists[j]
        for i in year:
            month_chl = monthly_integrated_chl(dataset,i,month_actual)
            monthly_chl.append(month_chl)
    if verbose is True:
        print("Successfully calculated the monthly integrated chlorophyll")

    data = {
        'Number_of_blooms': bloom_events_annual,
        'Number_of_blooms_biological': bloom_events_bio,
        'Total_bloom_days_calendar': bloom_days_annual,
        'Bloom_days_above_threshold_calendar': bloom_days_annual_above_thld,
        'Total_bloom_days_biological': bio_bloom_days_annual,
        'Bloom_days_above_threshold_biological': bio_bloom_days_above_thld,
        'Percent_total_bloom_days_above_threshold': percent_annual_thld,
        'Percent_total_bloom_days_above_threshold_biological': percent_bio_thld,
        'Total_integrated_chl': integrated_annual,
        'Bloom_period_integrated_chl': bloom_integrated_annual,
        'Biological_total_integrated': biological_integrated_annual,
        'Biological_bloom_period_integrated': biological_bloom_int_annual,
        'Bloom_year_integrated_chl': bloom_year_integrated,
        'Bloom_year_bloom_only_integrated': bloom_year_bloom_only,
        'January_integrated_chl': jan_chl,
        'February_integrated_chl': feb_chl,
        'March_integrated_chl': mar_chl,
        'April_integrated_chl': apr_chl,
        'May_integrated_chl': may_chl,
        'June_integrated_chl': jun_chl,
        'July_integrated_chl': jul_chl,
        'August_integrated_chl': aug_chl,
        'September_integrated_chl': sep_chl,
        'October_integrated_chl': oct_chl,
        'November_integrated_chl': nov_chl,
        'December_integrated_chl': dec_chl,
    }
    df = pd.DataFrame(data,index=year)
    df.index.name = "Year"
    if verbose is True:
        print("Successfully created dataframe")
    return df

### Save the bloom metrics as csv files for visualizing

In [ ]:
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
for x in range(5):
    df = summary_bloom_metrics(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_region[x])
    df.to_csv(rf'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Summary_Stats.csv',mode='w')
    print("Successfully saved file :)")

In [ ]:
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
for x in range(5):
    df = bloom_metrics(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_region[x])
    df.to_csv(rf'C:\Users\grace\OneDrive\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Bloom_Metrics.csv',mode='w')
    print("Successfully saved file :)")